# Etapa B3 · Repetir el tiempo de cómputo — Mac mini

La primera medición (notebook `faseB_complementarias`, paso 2) dio **(T2−T1)/T0 negativo** para MAREA-2,
L-SHADE, jSO y PSO: T1 se midió una sola vez y salió inflado. En el MacBook, T2 de PSO queda
justo encima de T1, como debe ser, así que no es un error del código sino del estado del mini en esa toma.

Ahora T1 se mide **antes de cada algoritmo** (10 tomas por dimensión) y se usa el mínimo. `verificar`
exige T2 > T1 en todos.

Copiar al mini (commit `0ffcf4f`): **`complementarias.py`**. `marea2.py`, `experimento_config.py` y
`competidores.py` no cambian; la celda 0 lo comprueba con hashes.

Tiempo aprox.: **~25 min, con la máquina libre** (sin otros notebooks ni procesos pesados).
**Solo en el Mac mini.**

In [ ]:
import sys, os, json, hashlib, subprocess, opfunu

RUNS_CONV = 51
ESPERADO = {
    "complementarias.py":    "2889b74aae020d143a4980e19a70ccc4c6e959dceb379f91e178c681cee8af1a",
    "marea2.py":             "10d7448e59082adf4f464068646d74d25d8b8f54f5317f9eb4dfed32dacfe64b",
    "experimento_config.py": "ef9c5af25d46afa681276efeb5d0e25c68290404dc6093e8d8783b3554a914d1",
    "competidores.py":       "783a7ada89ddf917332d91efcf8179a8c3c703797943eb415d14a7141c68487b",
}

def correr(*args):
    """Ejecuta un script con el mismo Python del kernel y muestra la salida en vivo."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        if "pkg_resources" not in linea:
            print(linea, end="")
    return p.wait()

print("python :", sys.executable)
print("carpeta:", os.getcwd())
print("opfunu :", opfunu.__version__)
assert opfunu.__version__ == "1.0.4", "opfunu tiene que ser 1.0.4 (1.0.1 esta roto)"
for f, h in ESPERADO.items():
    real = hashlib.sha256(open(f, "rb").read()).hexdigest()
    print(f"{f:24s} {real[:16]}  {'OK' if real == h else 'DISTINTO'}")
    assert real == h, f"{f} no es la version del commit 0ffcf4f: copiala de nuevo"
print("Entorno y archivos OK")

## 1 · Respaldar la medición anterior
Renombra `TIEMPO.json` a `TIEMPO_v1_T1unico.json`. Si el `TIEMPO.json` ya es del formato nuevo (una
corrida de este notebook interrumpida), no lo toca y el paso 2 reanuda.

In [ ]:
P = "resultados_experimento/complementarias/TIEMPO.json"
B = "resultados_experimento/complementarias/TIEMPO_v1_T1unico.json"
ALGS = ["MAREA2_A2_hgt", "LSHADE", "JSO", "DE", "PSO", "ABC", "GWO", "WOA", "FA", "CLONALG"]
if os.path.exists(P) and not any(f"T1_{a}_D{d}" in json.load(open(P)) for a in ALGS for d in (10, 50)):
    assert not os.path.exists(B), f"ya existe {B}: no sobrescribir, mandar esta salida a Claude"
    os.rename(P, B)
    print("respaldado en", B)
else:
    print("nada que respaldar (no hay TIEMPO.json viejo, o es una medicion nueva a medio camino)")
RESPALDADO = True

## 2 · Tiempo de cómputo, B3 (~25 min) — **con la máquina libre**
Secuencial. No correr otras celdas ni otros notebooks mientras tanto. Cada línea muestra la toma de T1
antes del algoritmo; tienen que salir parecidas entre sí.

In [ ]:
assert globals().get("RESPALDADO"), "primero la celda de respaldo"
rc = correr("complementarias.py", "tiempo")
assert rc == 0

## 3 · Verificar y traer el resultado al MacBook
Tiene que terminar en **TODO OK** (ahora incluye "T2 > T1 en todos"). Mandar también las dos líneas
`tiempo D10/D50: T1 entre ... y ...`.

Traer al MacBook **solo** `resultados_experimento/complementarias/TIEMPO.json`, a esa misma carpeta
(**no a la raíz** del repo).

In [ ]:
rc = correr("complementarias.py", "verificar", "--runs", str(RUNS_CONV))
print("\nverificacion:", "OK" if rc == 0 else "REVISAR")